# 03 — Análise de Churn e Scoring de Upsell
**Projeto:** churn-upsell-analytics · **Etapa 3 de 3**

Aqui geramos os gráficos finais, a tabela de KPIs e o arquivo para o Power BI.
Essa é a etapa que transforma dados em decisão de negócio.


## 1. Imports e carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_style("whitegrid")
sns.set_palette("muted")

PROJECT_ROOT = Path.cwd().parent
CLEAN_PATH   = PROJECT_ROOT / "data" / "processed" / "customers_clean.csv"
KPI_PATH     = PROJECT_ROOT / "data" / "processed" / "churn_kpi_processed.csv"
CHARTS_PATH  = PROJECT_ROOT / "docs" / "screenshots"
CHARTS_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN_PATH)
print(f"Carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

## 2. Taxa de churn por segmento demográfico (replicando SQL Query 1)

In [ ]:
# Igual à Query 1 do SQL, mas em Python — mesma lógica, ferramenta diferente
churn_matrix = (
    df.groupby(["age_segment", "balance_segment"], observed=True)["exited"]
      .agg(["count", "sum", "mean"])
      .rename(columns={"count": "total", "sum": "churned", "mean": "churn_rate"})
)
churn_matrix["churn_rate"] = (churn_matrix["churn_rate"] * 100).round(1)
churn_matrix = churn_matrix.sort_values("churn_rate", ascending=False)

print("Top 10 segmentos de maior risco de churn:")
print(churn_matrix.head(10).to_string())

## 3. Gráfico A — Heatmap de churn (idade × saldo)

In [ ]:
# Heatmap: visualização em grade colorida — quanto mais vermelho, maior o churn
pivot = df.pivot_table(
    index="age_segment", columns="balance_segment",
    values="exited", aggfunc="mean", observed=True
) * 100

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
            cbar_kws={"label": "Taxa de churn (%)"}, ax=ax, linewidths=0.5)
ax.set_title("Taxa de Churn: Idade × Saldo Bancário", fontsize=13, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(CHARTS_PATH / "churn_heatmap_age_balance.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Gráfico B — Churn por número de produtos

In [ ]:
# Replica a Query 4 do SQL — costuma revelar o padrão contraintuitivo:
# mais produtos ≠ mais fidelidade

prod_churn = df.groupby("num_of_products")["exited"].agg(["count", "mean"])
prod_churn.columns = ["total_customers", "churn_rate"]
prod_churn["churn_rate"] = (prod_churn["churn_rate"] * 100).round(1)

print(prod_churn.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#55A868" if v < 20 else "#DD8452" if v < 40 else "#C44E52"
          for v in prod_churn["churn_rate"]]
prod_churn["churn_rate"].plot(kind="bar", ax=ax, color=colors, edgecolor="white")
ax.set_title("Taxa de Churn por Número de Produtos", fontsize=12, pad=10)
ax.set_xlabel("Número de produtos contratados")
ax.set_ylabel("Taxa de churn (%)")
ax.tick_params(axis="x", rotation=0)

for i, v in enumerate(prod_churn["churn_rate"]):
    ax.text(i, v + 1, f"{v}%", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(CHARTS_PATH / "churn_by_num_products.png", dpi=150, bbox_inches="tight")
plt.show()

if prod_churn["churn_rate"].iloc[2:].mean() > prod_churn["churn_rate"].iloc[:2].mean():
    print("\n⚠ Achado relevante: clientes com 3-4 produtos têm MAIOR churn que")
    print("  clientes com 1-2. Isso geralmente indica venda cruzada mal ajustada")
    print("  ou insatisfação anterior ao cancelamento total.")

## 5. Gráfico C — Segmentos de upsell

In [ ]:
upsell_counts = df["upsell_segment"].value_counts()

colors_map = {
    "High Upsell Potential": "#55A868",
    "Medium Upsell Potential": "#DD8452",
    "Low Upsell Potential": "#8172B2",
    "N/A (Churned)": "#C44E52"
}
colors = [colors_map.get(x, "#999999") for x in upsell_counts.index]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(upsell_counts.index, upsell_counts.values, color=colors, edgecolor="white")
for bar, val in zip(bars, upsell_counts.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)

ax.set_title("Distribuição de Segmentos de Upsell", fontsize=12, pad=10)
ax.set_xlabel("Qtd. de clientes")
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "upsell_segments.png", dpi=150, bbox_inches="tight")
plt.show()

n_high = (df["upsell_segment"] == "High Upsell Potential").sum()
print(f"\n{n_high:,} clientes identificados como 'High Upsell Potential'")
print("→ Esses são os candidatos prioritários para oferta de segundo produto")

## 6. Gráfico D — Churn por país e por ativação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

geo_churn = (df.groupby("geography")["exited"].mean() * 100).sort_values(ascending=False)
axes[0].bar(geo_churn.index, geo_churn.values, color="#4C72B0", edgecolor="white")
axes[0].set_title("Churn por País", fontsize=12)
axes[0].set_ylabel("Taxa de churn (%)")
for i, v in enumerate(geo_churn.values):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)

active_churn = (df.groupby("is_active_member")["exited"].mean() * 100)
active_churn.index = ["Inativo", "Ativo"]
axes[1].bar(active_churn.index, active_churn.values, color=["#C44E52", "#55A868"], edgecolor="white")
axes[1].set_title("Churn: Cliente Ativo vs Inativo", fontsize=12)
axes[1].set_ylabel("Taxa de churn (%)")
for i, v in enumerate(active_churn.values):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(CHARTS_PATH / "churn_geo_activity.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Tabela de KPIs consolidada — exportar para Power BI

In [ ]:
kpi = (
    df.groupby(["geography", "age_segment"], observed=True)
      .agg(
          total_customers = ("customer_id", "count"),
          churn_rate_pct  = ("exited", lambda x: round(x.mean()*100, 1)),
          avg_balance     = ("balance", lambda x: round(x.mean(), 2)),
          avg_products    = ("num_of_products", lambda x: round(x.mean(), 2)),
          high_upsell_n   = ("upsell_segment", lambda x: (x == "High Upsell Potential").sum())
      )
      .reset_index()
)

KPI_PATH.parent.mkdir(parents=True, exist_ok=True)
kpi.to_csv(KPI_PATH, index=False)

print(f"KPI exportado: {KPI_PATH}")
print(kpi.to_string(index=False))

## 8. Resumo executivo

In [ ]:
churn_geral   = df["exited"].mean() * 100
n_high_upsell = (df["upsell_segment"] == "High Upsell Potential").sum()
n_total       = len(df)

print("=" * 55)
print("  CHURN & UPSELL ANALYTICS — RESUMO EXECUTIVO")
print("=" * 55)
print(f"  Base de clientes         : {n_total:,}")
print(f"  Taxa de churn geral      : {churn_geral:.1f}%")
print(f"  Maior risco (segmento)   : {churn_matrix.index[0]}")
print(f"  Candidatos a upsell      : {n_high_upsell:,} ({n_high_upsell/n_total*100:.1f}% da base)")
print()
print("  Ação recomendada:")
print("  1. Priorizar retenção nos segmentos de maior churn (heatmap)")
print("  2. Revisar política de venda cruzada para 3+ produtos")
print("  3. Direcionar oferta de 2º produto para o grupo High Upsell")
print("=" * 55)
print()
print("Gráficos gerados em docs/screenshots/:")
for f in sorted(CHARTS_PATH.glob("*.png")):
    print(f"  {f.name}")